# 08 — Statistiques des capteurs et actionneurs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/SerreIA/blob/main/notebooks/08_stats_capteurs.ipynb)

Séries temporelles de la serre (docs/19 §5) : couverture et trous, qualité,
distributions et valeurs aberrantes, profils journaliers, DLI, autocorrélations
(horizon de prévision), corrélations, cycles des actionneurs et délai commande → effet,
alertes et vision. Source : un scénario figé (`data/scenarios/<nom>/serre.db`, figé à la
volée s'il manque), l'historique simulé de `sim.dataset.generate` ou une base SQLite.
Calcul dans `sim/stats.py` ; sorties dans `reports/datasets/capteurs/<source>/`.

## Paramètres

In [ ]:
SOURCE = "scenario:journee"  # "scenario:<nom>", "sim" ou chemin d'une base SQLite
SIM_DAYS = 30  # SOURCE = "sim" : jours simulés (pas de 10 min)
SIM_SEED = 0
GAP_FACTOR = 2  # trou : écart > GAP_FACTOR × la cadence médiane du capteur
FUSEAU = "Europe/Paris"  # heure locale des profils ; None : celui de la table serre
PANNES = []  # sim/faults.py, ex. ["fige:z1/temperature@30-90", "deconnexion:esp32-02@10-40"]
DATASETS = []  # aucune archive Drive : scénarios figés à la volée, `sim` généré

## Environnement

In [ ]:
# Environnement (PC et Colab) : dépôt, données, sorties (docs/data-drive.md).
# PC : rien n'est installé ; un jeu absent est récupéré du Drive par rclone, sinon la cellule
# s'arrête avec la commande à lancer. Colab : clone du dépôt à REV, installation, données via
# le Drive monté (archives MyDrive/SerreIA/data/*.tar), sorties précédentes restaurées.
import logging
import os
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/SerreIA"  # Colab : data/*.tar et runs/ ; None : pas de Drive
REPO_URL = "https://github.com/lrivals/SerreIA.git"
REV = "main"
COLAB = "google.colab" in sys.modules

if COLAB:
    ROOT = Path("/content/SerreIA")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[vision,controle,backend]"], check=True
    )
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from tools.notebooks import drive_data  # noqa: E402

if COLAB and DRIVE_DIR:
    drive_data.mount_drive()
    drive_data.restore_all(DRIVE_DIR)
drive_data.ensure(DATASETS, DRIVE_DIR if COLAB else None)

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
ENV_SET = (
    [f"data.num_workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)

## Données

In [ ]:
import re

import pandas as pd
from IPython.display import Markdown, display

from common.report import REPORTS_DIR, write_report
from sim import stats as ss
from sim import stats_plots as sp

src = ss.load_source(SOURCE, SIM_DAYS, SIM_SEED)
L = ss.inject_faults(src.mesures, PANNES)  # mesures en format long
TZ = FUSEAU or src.serre.get("fuseau", "Europe/Paris")
STEP = "10min"
W = ss.wide(L, STEP)  # table large (zone, type), pas régulier
OUT = REPORTS_DIR / "capteurs" / re.sub(r"[^\w-]+", "_", SOURCE).strip("_")
OUT.mkdir(parents=True, exist_ok=True)
SUMMARY = {"source": SOURCE, "pannes": PANNES, "debut": L["ts"].min(),
           "fin": L["ts"].max(), "n_mesures": len(L),
           "n_capteurs": L["capteur_id"].nunique(), "n_etats": len(src.etats),
           "fuseau": TZ}
SECTIONS = {}
display(pd.Series(SUMMARY).to_frame("valeur"))

## 5.1 Couverture et qualité

Trou : écart entre deux mesures > `GAP_FACTOR` × la cadence médiane du capteur. Les
scénarios n'ont que des mesures `ok` : cela ne prouve pas l'absence de défaut ;
renseigner `PANNES` pour vérifier que ces statistiques révèlent les pannes simulées.

In [ ]:
cov = ss.coverage(L, GAP_FACTOR)
display(cov)
gaps = ss.gaps(L, GAP_FACTOR)
display(sp.plot_timeline(L, gaps, OUT / "couverture.png"))
share, only_ok = ss.quality_share(L)
display(share)
if only_ok:
    print("Toutes les mesures sont `ok` : aucun défaut signalé, ce qui ne veut pas dire "
          "qu'il n'y en a pas (injecter des PANNES).")
SUMMARY.update(trous=int(cov["n_trous"].sum()), trou_max_s=float(cov["trou_max_s"].max()),
               que_des_ok=only_ok)
SECTIONS["Couverture"] = cov

## 5.2 Distributions

Résumés par type × zone, valeurs hors bornes physiques (`ss.PHYSICAL_BOUNDS`),
saturation du capteur de lumière, valeurs aberrantes par z-score robuste (médiane,
MAD) sur fenêtre glissante d'une heure.

In [ ]:
display(ss.summaries(L))
display(sp.plot_histograms(L, OUT / "histogrammes.png"))
oob = ss.out_of_bounds(L)
print(f"{len(oob)} valeurs hors bornes physiques")
display(oob.groupby(["zone", "type"]).size().rename("n").to_frame())
display(ss.saturation(L))
outl = ss.robust_outliers(L)
print(f"{len(outl)} valeurs aberrantes (|z robuste| > 5)")
display(outl.groupby(["zone", "type"]).size().rename("n").to_frame())
SUMMARY.update(hors_bornes=len(oob), aberrantes=len(outl))
SECTIONS["Résumés"] = ss.summaries(L)

## 5.3 Dynamique temporelle

Profils journaliers en heure locale (médiane et IQR), DLI (même conversion lux → PPFD
que la serre), autocorrélations : l'horizon où l'ACF passe sous 0,5 borne ce que la
persistance prévoit seule (niveau 2, docs/11). Décomposition tendance / journalier /
résidu, corrélations entre capteurs d'une zone.

In [ ]:
prof = ss.daily_profile(L, ["temperature", "humidite", "lux", "co2"], TZ)
display(sp.plot_daily_profiles(prof, OUT / "profils_journaliers.png"))
dli = ss.dli(L, src.serre.get("lux_vers_ppfd", ss.LUX_VERS_PPFD), TZ)
display(dli)
acfs = ss.autocorrelations(W, "24h")
horizon = ss.useful_horizon(acfs)
display(horizon.rename("horizon ACF < 0,5 (h)").to_frame())
display(sp.plot_acf(acfs, path=OUT / "autocorrelations.png"))
SUMMARY.update(dli=dli.to_dict(orient="records"),
               horizon_h={" ".join(k): v for k, v in horizon.items()})
SECTIONS["Horizon ACF < 0,5 (h)"] = horizon.to_frame("h")

In [ ]:
for zone in sorted(W.columns.get_level_values(0).unique()):
    if (zone, "temperature") in W.columns:
        comps, parts = ss.decompose(W[(zone, "temperature")])
        print(zone, "température :", parts)
        display(sp.plot_decomposition(comps, f"{zone} température",
                                      OUT / f"decomposition_{zone}.png"))
    corr = ss.correlations(W, zone)
    display(sp.plot_corr(corr, zone, OUT / f"correlations_{zone}.png"))

## 5.4 Actionneurs, alertes, vision

Taux d'activation et durées des cycles ; délai commande → effet : décalage du pic de
corrélation croisée entre l'état d'un actionneur et la dérivée du capteur visé
(ventilateur → T°, brumisateur → HR, pompe → humidité du sol).

In [ ]:
duty = ss.duty_cycles(src.etats)
display(duty)
display(sp.plot_states(src.etats, path=OUT / "actionneurs.png"))
lag = ss.lag_effect(src.etats, L)
display(lag)
SUMMARY.update(taux_activation={" ".join(k): v for k, v in
                                duty["taux_activation"].items()},
               delai_effet=lag.to_dict(orient="records"))
SECTIONS["Actionneurs"] = duty
SECTIONS["Délai commande → effet"] = lag

In [ ]:
al = ss.alerts_summary(src.alertes)
print(f"{al['n']} alertes")
if al["n"]:
    display(al["niveau_zone"])
    display(al["par_source"].to_frame())
    display(sp.plot_alerts(src.alertes, OUT / "alertes.png"))
vi = ss.vision_summary(src.vision)
print(f"{vi['n']} analyses vision")
if vi["n"]:
    display(vi["par_zone"].to_frame())
    display(vi["sante"])
    display(vi["confiances"])
SUMMARY.update(alertes=al["n"], analyses_vision=vi["n"])

## Rapport

In [ ]:
write_report(OUT, SUMMARY, SECTIONS)
drive_data.sync_all(DRIVE_DIR)
display(Markdown((OUT / "summary.md").read_text(encoding="utf-8")))

## Résumé

`reports/datasets/capteurs/<source>/` : `summary.json`, `summary.md`, figures.
Décisions (docs/19 §9) : horizon de prévision du niveau 2 au vu des autocorrélations,
seuils de trous (`capteur_muet_s`) et de valeurs aberrantes du contrôleur. Ce notebook
passe en P1 dès que la base réelle de la serre existe (`SOURCE` = son chemin).